<a href="https://colab.research.google.com/github/ms914/3DKWM_server/blob/main/kwm_cli/KWM_CLI_Mehrfachbindungen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://share.gemini.google/vVXC6mxD25Cp

In [1]:
!pip install clifford

In [2]:
code = """
from fastapi import FastAPI
import numpy as np
from clifford.g3c import *
import uvicorn

app = FastAPI()
WELT_ZUSTAND = {}

# --- CGA MATHEMATIK-ENGINE ---
def cga_punkt(x, y, z):
    return up(x*e1 + y*e2 + z*e3)

def berechne_richtungs_rotor(v_start, v_ziel):
    \"\"\"Bringt Vektor v_start zur Deckung mit v_ziel\"\"\"
    v_s = (v_start[0]*e1 + v_start[1]*e2 + v_start[2]*e3).normal()
    v_z = (v_ziel[0]*e1 + v_ziel[1]*e2 + v_ziel[2]*e3).normal()
    gp = v_z * v_s
    return (1 + gp).normal()

def berechne_axialen_rotor(achse_vec, winkel_rad):
    \"\"\"Erzeugt eine Drehung um eine feste Achse (Bivektor-Exponentialform)\"\"\"
    B = (achse_vec[0]*e1 + achse_vec[1]*e2 + achse_vec[2]*e3).normal() * (e1*e2*e3)
    return np.cos(winkel_rad / 2) - np.sin(winkel_rad / 2) * B

# --- PARAMETER ---
# --- PARAMETER ERWEITERT ---
ELEMENT_PROFIL = {
    "H":  {"geo": "KUGEL",     "valenzelektronen": 1, "abstand": 1.10},
    "C":  {"geo": "TETRAEDER", "valenzelektronen": 4, "abstand": 1.25},
    "O":  {"geo": "TETRAEDER", "valenzelektronen": 6, "abstand": 1.20}, # O-H Abstand ca. 1.20 in unserem Gitter
}

def hole_lokale_wolken(symbol):
    if symbol == "H": return [[0.0, 0.0, 0.0]]
    if symbol in ["C", "O"]:
        # Beide nutzen die tetraedrischen Achsen
        return [
            [1/np.sqrt(3), 1/np.sqrt(3), 1/np.sqrt(3)],
            [-1/np.sqrt(3), -1/np.sqrt(3), 1/np.sqrt(3)],
            [-1/np.sqrt(3), 1/np.sqrt(3), -1/np.sqrt(3)],
            [1/np.sqrt(3), -1/np.sqrt(3), -1/np.sqrt(3)]
        ]
    return [[0.0, 0.0, 0.0]]

@app.get("/status")
def get_status():
    return WELT_ZUSTAND

@app.post("/erzeuge")
def erzeuge(daten: dict):
    symbol = daten["symbol"]
    uid = f"{symbol}_{len(WELT_ZUSTAND) + 1}"

    # Valenzelektronen auf die 4 Wolken verteilen
    wolken = []
    lokale_achsen = hole_lokale_wolken(symbol)

    if symbol == "O":
        # Sauerstoff: 6 Elektronen -> zwei Wolken haben 2 (voll), zwei haben 1 (reaktiv)
        elektronen_verteilung = [2, 2, 1, 1]
    elif symbol == "C":
        elektronen_verteilung = [1, 1, 1, 1]
    else: # H
        elektronen_verteilung = [1]

    WELT_ZUSTAND[uid] = {
        "symbol": symbol,
        "position": (np.random.rand(3) - 0.5 * 2).tolist(),
        "rotor_bivektor": [0.0, 0.0, 0.0],
        "wolken": [{"id": i, "elektronen": e, "lokal": v} for i, (e, v) in enumerate(zip(elektronen_verteilung, lokale_achsen))]
    }
    return {"status": "success", "uid": uid}

@app.post("/binde")
def binde(daten: dict):
    if daten["atom_A_id"] not in WELT_ZUSTAND or daten["atom_B_id"] not in WELT_ZUSTAND:
        return {"status": "error", "message": "Eines der Atome existiert nicht. ID prüfen!"}

    atom_A = WELT_ZUSTAND[daten["atom_A_id"]]
    atom_B = WELT_ZUSTAND[daten["atom_B_id"]]

    for w in atom_A["wolken"]:
        if "partner" not in w: w["partner"] = None
    for w in atom_B["wolken"]:
        if "partner" not in w: w["partner"] = None

    bereits_gebunden = sum(1 for w in atom_A["wolken"] if w["partner"] == daten["atom_B_id"])
    ziel_ordnung = bereits_gebunden + 1

    if ziel_ordnung > 3:
        return {"status": "error", "message": "Maximale Bindungsordnung erreicht."}

    # Alte Sortierung nimmt auch existierende nicht bindende Wolken mit 2 Elektronen an
    # freie_A = [w for w in atom_A["wolken"] if w["elektronen"] < 3]
    # freie_B = [w for w in atom_B["wolken"] if w["elektronen"] < 3]

    # Sortiere so, dass Wolken mit 1 Elektron (Radikale) VOR Wolken mit 2 Elektronen (Paare) kommen
    freie_A = sorted([w for w in atom_A["wolken"] if w["elektronen"] < 3], key=lambda x: x["elektronen"])
    freie_B = sorted([w for w in atom_B["wolken"] if w["elektronen"] < 3], key=lambda x: x["elektronen"])

    if not freie_A or not freie_B:
        return {"status": "error", "message": "Keine freien Wolken verfügbar."}

    # Nutze spezifischen C-H Bindungsabstand aus dem Profil

    if "O" in [atom_A["symbol"], atom_B["symbol"]]:
        abstand = ELEMENT_PROFIL["O"]["abstand"]
    else:
        abstand = ELEMENT_PROFIL["H"]["abstand"] if (atom_A["symbol"] == "H" or atom_B["symbol"] == "H") else ELEMENT_PROFIL["C"]["abstand"]




    # 3. GEOMETRISCHE ACHSE (Translation - Jetzt voll dynamisch)
    lokaler_vektor_A = np.array(freie_A[0]["lokal"])
    if np.linalg.norm(lokaler_vektor_A) == 0:
        achse_A = np.array([1.0, 0.0, 0.0])
    else:
        achse_A = lokaler_vektor_A / np.linalg.norm(lokaler_vektor_A)

    neue_pos_B = np.array(atom_A["position"]) + (achse_A * abstand)
    atom_B["position"] = neue_pos_B.tolist()

    # 4. CGA ROTOR-KASKADE (Hier war der Bug-Fix!)
    ziel_achse_B = -achse_A
    start_achse_B = np.array(freie_B[0]["lokal"])
    if np.linalg.norm(start_achse_B) == 0:
        start_achse_B = np.array([1.0, 0.0, 0.0])
    start_achse_B = start_achse_B / np.linalg.norm(start_achse_B)

    # JETZT REPARIERT: start_achse_B wird korrekt übergeben
    R_final = berechne_richtungs_rotor(start_achse_B, ziel_achse_B)

    if ziel_ordnung == 2:
        R_torsion = berechne_axialen_rotor(achse_A, np.radians(0))
        R_final = R_torsion * R_final
    elif ziel_ordnung == 3:
        R_torsion = berechne_axialen_rotor(achse_A, np.radians(60))
        R_final = R_torsion * R_final

    atom_B["rotor_bivektor"] = [float(R_final[e1*e2]), float(R_final[e2*e3]), float(R_final[e3*e1])]

    # 5. STATUS UPDATEN
    freie_A[0]["elektronen"] = 3
    freie_A[0]["partner"] = daten["atom_B_id"]
    freie_B[0]["elektronen"] = 3
    freie_B[0]["partner"] = daten["atom_A_id"]

    return {"status": "success", "bindung_aktuell": ziel_ordnung}
"""

with open("server.py", "w") as f:
    f.write(code)

import subprocess
import time
subprocess.run(["pkill", "-f", "uvicorn"])
time.sleep(1)
subprocess.Popen(["uvicorn", "server:app", "--port", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("⚡ Reparierter CGA-Server läuft stabil im Hintergrund!")

⚡ Reparierter CGA-Server läuft stabil im Hintergrund!


In [ ]:
import requests
import sys

class CgaCliController:
    def __init__(self, base_url="http://127.0.0.1:8000"):
        self.base_url = base_url

    def hole_zustand(self):
        try:
            return requests.get(f"{self.base_url}/status").json()
        except Exception as e:
            print(f" Fehler beim Server-Abruf: {e}")
            return {}

    def erzeuge(self, symbol):
        try:
            return requests.post(f"{self.base_url}/erzeuge", json={"symbol": symbol}).json()
        except:
            return {"status": "error"}

    def binde(self, id_a, id_b):
        try:
            return requests.post(f"{self.base_url}/binde", json={"atom_A_id": id_a, "atom_B_id": id_b}).json()
        except:
            return {"status": "error", "message": "Verbindung zum Server fehlgeschlagen."}

def zeige_welt(welt):
    print("\n" + "="*95)
    print(" CGA-MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)")
    print("="*95)

    if not welt:
        print(" Keine Atome in der Szene. Nutze 'erzeuge <C/H>' um zu starten.")
        print("="*95)
        return

    for uid, data in welt.items():
        pos = [f"{x:.2f}" for x in data.get('position', [0,0,0])]

        # Sammler für die Lewis-Symbole dieses Atoms
        freie_elektronen = 0
        freie_paare = 0
        bindungen_partner = {} # Speichert, wie viele Bindungen zu welchem Partner existieren

        for w in data.get('wolken', []):
            anzahl_e = w.get('elektronen', 0)
            partner = w.get('partner')

            if anzahl_e == 3 and partner: # Gebundene Wolke
                bindungen_partner[partner] = bindungen_partner.get(partner, 0) + 1
            elif anzahl_e == 2:
                freie_paare += 1
            elif anzahl_e == 1:
                freie_elektronen += 1

        # 1. Standard-Zustand der Wolken (KWM-Rohdaten)
        wolken_raw = []
        for w in data.get('wolken', []):
            if w.get('elektronen') == 3: wolken_raw.append("≡" if bindungen_partner.get(w.get('partner'), 1) == 3 else ("═" if bindungen_partner.get(w.get('partner'), 1) == 2 else "─"))
            elif w.get('elektronen') == 2: wolken_raw.append("|")
            else: wolken_raw.append("•")
        wolken_display = " ".join(wolken_raw)

        # 2. Chemische Lewis-Kompaktformel generieren (z.B. O mit zwei Strichen und zwei Punkten -> |O• •|)
        lewis_links = "|" * freie_paare
        lewis_rechts = "|" * freie_paare if freie_paare > 1 else ""
        lewis_punkte = "•" * freie_elektronen

        # Bindungspartner-Striche auflisten
        lewis_bindungen = []
        for p_id, ordnung in bindungen_partner.items():
            strich = "─" if ordnung == 1 else ("═" if ordnung == 2 else "≡")
            lewis_bindungen.append(f"{strich}{p_id}")
        bindungs_str = " ".join(lewis_bindungen)

        # Zusammenbau der Lewis-Schreibweise
        symbol = data['symbol']
        lewis_formel = f"{lewis_links}{symbol}{lewis_punkte}{lewis_rechts} {bindungs_str}".strip()

        print(f"ID: {uid:<6} | Pos: {str(pos):<22} | KWM: [ {wolken_display} ] | Lewis: {lewis_formel}")
    print("="*95)

# --- REALE CLI INTERAKTIONS-SCHLEIFE ---
cc = CgaCliController()

print("CLI-Frontend erfolgreich geladen.")
while True:
    aktueller_zustand = cc.hole_zustand()
    zeige_welt(aktueller_zustand)

    try:
        eingabe = input("Befehl [erzeuge <C/H> | binde <id1> <id2> | exit]: ").strip().split()
    except (KeyboardInterrupt, EOFError):
        print("\nCLI beendet.")
        break

    if not eingabe or eingabe[0] == "exit":
        print("CLI beendet.")
        break

    kommando = eingabe[0].lower()



    if kommando == "erzeuge" and len(eingabe) > 1:
        symbol = eingabe[1].upper()
        if symbol in ["C", "H", "O"]: # 'O' hinzugefügt!
            res = cc.erzeuge(symbol)
            if res.get("status") == "success":
                print(f"-> Erfolg: {res['uid']} im 3D-Raum platziert.")
        else:
            print("-> Fehler: Nur 'C' (Kohlenstoff) und 'H' (Wasserstoff) werden im Profil unterstützt.")

    elif kommando == "binde" and len(eingabe) > 2:
        id_a, id_b = eingabe[1], eingabe[2]
        if id_a == id_b:
            print("-> Fehler: Ein Atom kann sich nicht mit sich selbst verbinden.")
            continue

        res = cc.binde(id_a, id_b)
        if res.get("status") == "success":
            print(f"-> Erfolg! Bindungsstufe erhöht auf: {res['bindung_aktuell']}")
        else:
            print(f"-> Fehler vom Server: {res.get('message', 'Unbekannter Fehler')}")

    else:
        print("-> Unbekannter Befehl oder falsche Anzahl an Argumenten.")

CLI-Frontend erfolgreich geladen.
 Fehler beim Server-Abruf: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /status (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7830f64c5220>: Failed to establish a new connection: [Errno 111] Connection refused'))

 CGA-MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)
 Keine Atome in der Szene. Nutze 'erzeuge <C/H>' um zu starten.
Befehl [erzeuge <C/H> | binde <id1> <id2> | exit]: erzeuge C
-> Erfolg: C_1 im 3D-Raum platziert.

 CGA-MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)
ID: C_1    | Pos: ['-0.24', '-0.08', '-0.99'] | KWM: [ • • • • ] | Lewis: C••••
Befehl [erzeuge <C/H> | binde <id1> <id2> | exit]: erzeuge C
-> Erfolg: C_2 im 3D-Raum platziert.

 CGA-MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)
ID: C_1    | Pos: ['-0.24', '-0.08', '-0.99'] | KWM: [ • • • • ] | Lewis: C••••
ID: C_2    | Pos: ['-0.81', '-0.03', '-0.48'] | KWM: [ • • • • ] | Lewis: C••••
Befehl [erzeuge <C/H> | binde <id1> <id2> | exit]: 